B"H

# Term Project: Milestone 3: Model building and evaluation

DSC-550

David Koyrakh

Professor Brett Werner


## Introduction

For Milestone 3, I continue my term project of predicting next-day Bitcoin price movement (up or down) using Google Trends and historical In Mielstones 1 and 2, I performed some initial EDA and prepared the data by cleaning, merging, and engineering features from both the Bitcoin price history and Google Trends datasets.

The goal of this milestone is to build and evaluate my first predictive model. I plan to:

...

My goal is to develop a model that can predict next-day Bitcoin price movement (up/down) with better-than-random accuracy. I will evaluate the model's performance using metrics appropriate for the model I end up choosing to build.

## Load the data

In [1]:
import pandas as pd
import numpy as np

# Load BTC data from json
btc_data_cleaned_loaded = pd.read_json(
    'btc_data_cleaned.json',
    convert_dates=['Date'] # Ensure dates are parsed correctly
)

# Load Google Trends data from json
filename = "data/2025-02-01_22-36-52/google_trends_aggregate_2015-01-23_to_2025-01-23_90b95999-201b-4f9e-a6bd-e0c16972826f.json"
trends_data_loaded = pd.read_json(filename, lines=True)

View the head of each dataset:

In [2]:
print("Historic BTC data:")
btc_data_cleaned_loaded.head()

Historic BTC data:


,Date,Close,pct_change_1d,sma_3d,volatility_3d,Price_up_tomorrow
0,2015-01-23,232.878998,NaN,NaN,NaN,0
1,2015-01-24,247.847000,0.064274,NaN,NaN,1
2,2015-01-25,253.718002,0.023688,244.814667,NaN,1
3,2015-01-26,273.472992,0.077862,258.345998,0.028186,0
4,2015-01-27,263.475006,-0.036559,263.555333,0.057238,0


In [3]:
print("Google Trends data:")
trends_data_loaded.head()

Google Trends data:


,date,bitcoin wallet,crypto wallet,buy crypto,bitget,buy bitcoin,isPartial,epoch_s,date_12am_gmt,request_id
0,2015-01-23,59,0,0,0,57,False,1421971200,01-23-2015,1
1,2015-01-24,62,0,0,0,60,False,1422057600,01-24-2015,1
2,2015-01-25,75,0,0,0,87,False,1422144000,01-25-2015,1
3,2015-01-26,57,0,0,0,83,False,1422230400,01-26-2015,1
4,2015-01-27,65,0,0,0,56,False,1422316800,01-27-2015,1


Both cleaned datasets have been loaded.

## Preprocess and merge the datasets

### Initial pipeline

Before proceeding to model building, there are several preprocessing steps that must be done. These will ensure that the data is actually fully clean and ready for modeling. Additional, I will add a few more features, such as Bitcoin technical indicators. Finally, I will merge the datasets into a single dataset ready for modeling.

To accomplish this preprocessing step, I have developed a `PreprocessingPipeline` class which accepts both datasets as initial arguments and outputs the fully processed and merged data set. This pipeline:
- Cleans Google Trends data by removing unnecessary columns and handling duplicate dates
- Validates and aligns timestamps between both datasets, ensuring proper date matching
- Adds technical indicators to the price data including:
  - RSI (14-day)
  - MACD with signal line and histogram
  - Bollinger Bands (20-day, 2 standard deviations)
- Adds price pattern features including:
  - Price momentum over 3, 7, and 14-day windows
  - Weekend indicators
  - Previous day's prediction accuracy
- Engineers Google Trends features including:
  - Rolling means (3 and 7-day) for each search term
  - Search momentum indicators
  - Aggregate search interest
- Handles missing values using forward-fill strategy with backward-fill for remaining gaps
- Handles extreme values using IQR-based outlier detection and capping
- Scales numerical features using StandardScaler while preserving certain columns (dates, target variable)
- Merges both datasets on the date column, ensuring proper alignment

The pipeline includes comprehensive error handling and logging to track the preprocessing steps and any potential issues that arise during execution.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from typing import Tuple, List, Dict
import logging

class PreprocessingPipeline:
    def __init__(self, price_df: pd.DataFrame, trends_df: pd.DataFrame):
        """
        Initialize the preprocessing pipeline.
        
        Args:
            price_df: DataFrame with Bitcoin price data
            trends_df: DataFrame with Google Trends data
        """
        self.price_df = price_df.copy()
        self.trends_df = trends_df.copy()
        self.feature_correlations = None
        self.removed_features = []
        self.scaler = StandardScaler()
        
        # Configure logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

    def clean_trends_data(self) -> None:
        """Clean and prepare the Google Trends data."""
        try:
            # Drop unnecessary columns
            columns_to_drop = ['isPartial', 'epoch_s', 'date_12am_gmt', 'request_id']
            self.trends_df.drop(columns=columns_to_drop, inplace=True)
            
            # Handle duplicate dates by taking the mean of duplicated values
            self.trends_df = self.trends_df.groupby('date').mean().reset_index()
            
            self.logger.info("Trends data cleaned successfully")
            
        except Exception as e:
            self.logger.error(f"Error cleaning trends data: {str(e)}")
            raise
        
    def validate_timestamps(self) -> Tuple[bool, str]:
        """
        Validate timestamps and align datasets, handling gaps appropriately.
        
        Returns:
            Tuple[bool, str]: (Success status, Information message)
        """
        try:
            # Convert date columns to datetime if they aren't already
            self.price_df['Date'] = pd.to_datetime(self.price_df['Date'])
            self.trends_df['date'] = pd.to_datetime(self.trends_df['date'])
            
            # Find the overlapping date range
            start_date = max(self.price_df['Date'].min(), self.trends_df['date'].min())
            end_date = min(self.price_df['Date'].max(), self.trends_df['date'].max())
            
            # Filter both datasets to the overlapping range
            self.price_df = self.price_df[
                (self.price_df['Date'] >= start_date) & 
                (self.price_df['Date'] <= end_date)
            ].copy()
            
            self.trends_df = self.trends_df[
                (self.trends_df['date'] >= start_date) & 
                (self.trends_df['date'] <= end_date)
            ].copy()
            
            # Check for any remaining gaps
            price_dates = set(self.price_df['Date'])
            trends_dates = set(self.trends_df['date'])
            
            missing_in_price = trends_dates - price_dates
            missing_in_trends = price_dates - trends_dates
            
            if missing_in_price or missing_in_trends:
                message = []
                if missing_in_price:
                    message.append(f"Missing {len(missing_in_price)} dates in price data")
                if missing_in_trends:
                    message.append(f"Missing {len(missing_in_trends)} dates in trends data")
                self.logger.warning(f"Data gaps found: {', '.join(message)}")
            
            # Create a complete date range and reindex both datasets
            full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
            
            # Reindex price data
            self.price_df.set_index('Date', inplace=True)
            self.price_df = self.price_df.reindex(full_dates)
            self.price_df.index.name = 'Date'
            self.price_df.reset_index(inplace=True)
            
            # Reindex trends data
            self.trends_df.set_index('date', inplace=True)
            self.trends_df = self.trends_df.reindex(full_dates)
            self.trends_df.index.name = 'date'
            self.trends_df.reset_index(inplace=True)
            
            return True, f"Timestamps aligned from {start_date} to {end_date}"
            
        except Exception as e:
            return False, f"Timestamp validation error: {str(e)}"
    
    def add_technical_indicators(self) -> None:
        """Add technical analysis indicators to the price dataset."""
        try:
            # RSI (14-day)
            delta = self.price_df['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            self.price_df['rsi_14'] = 100 - (100 / (1 + rs))
            
            # MACD
            exp1 = self.price_df['Close'].ewm(span=12, adjust=False).mean()
            exp2 = self.price_df['Close'].ewm(span=26, adjust=False).mean()
            self.price_df['macd'] = exp1 - exp2
            self.price_df['macd_signal'] = self.price_df['macd'].ewm(span=9, adjust=False).mean()
            self.price_df['macd_hist'] = self.price_df['macd'] - self.price_df['macd_signal']
            
            # Bollinger Bands (20-day, 2 standard deviations)
            self.price_df['bb_middle'] = self.price_df['Close'].rolling(window=20).mean()
            bb_std = self.price_df['Close'].rolling(window=20).std()
            self.price_df['bb_upper'] = self.price_df['bb_middle'] + (bb_std * 2)
            self.price_df['bb_lower'] = self.price_df['bb_middle'] - (bb_std * 2)
            self.price_df['bb_distance'] = (
                (self.price_df['Close'] - self.price_df['bb_middle']) / 
                (self.price_df['bb_upper'] - self.price_df['bb_lower'])
            )
            
            self.logger.info("Technical indicators added successfully")
            
        except Exception as e:
            self.logger.error(f"Error adding technical indicators: {str(e)}")
            raise
    
    def add_price_patterns(self) -> None:
        """Add price pattern indicators to the dataset."""
        try:
            # Price momentum over different timeframes
            for window in [3, 7, 14]:
                self.price_df[f'momentum_{window}d'] = self.price_df['Close'].pct_change(periods=window)
            
            # Weekend indicator
            self.price_df['is_weekend'] = self.price_df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
            
            # Previous day's prediction accuracy
            self.price_df['prev_prediction_correct'] = (
                (self.price_df['Price_up_tomorrow'].shift(1) == 1) & 
                (self.price_df['pct_change_1d'] > 0) |
                (self.price_df['Price_up_tomorrow'].shift(1) == 0) & 
                (self.price_df['pct_change_1d'] < 0)
            ).astype(int)
            
            self.logger.info("Price patterns added successfully")
            
        except Exception as e:
            self.logger.error(f"Error adding price patterns: {str(e)}")
            raise
    
    def add_trends_features(self) -> None:
        """Add engineered features from Google Trends data."""
        try:
            search_terms = ['bitcoin wallet', 'crypto wallet', 'buy crypto', 'bitget', 'buy bitcoin']
            
            # Calculate rolling means for each search term
            for term in search_terms:
                for window in [3, 7]:
                    self.trends_df[f'{term}_sma_{window}d'] = (
                        self.trends_df[term].rolling(window=window).mean()
                    )
                
                # Search momentum
                self.trends_df[f'{term}_momentum'] = self.trends_df[term].pct_change()
            
            # Aggregate search interest
            self.trends_df['aggregate_interest'] = self.trends_df[search_terms].mean(axis=1)
            
            self.logger.info("Trends features added successfully")
            
        except Exception as e:
            self.logger.error(f"Error adding trends features: {str(e)}")
            raise
    
    def handle_missing_values(self, strategy: str = 'forward_fill') -> Dict[str, int]:
        """
        Handle missing values in both datasets.
        
        Args:
            strategy: Strategy for handling missing values ('forward_fill', 'backward_fill', or 'drop')
            
        Returns:
            Dictionary with count of missing values handled per column
        """
        try:
            missing_counts = {}
            
            # Count initial missing values
            for df in [self.price_df, self.trends_df]:
                for column in df.columns:
                    missing_counts[column] = df[column].isna().sum()
            
            # Handle missing values according to strategy
            if strategy == 'forward_fill':
                self.price_df = self.price_df.ffill()
                self.trends_df = self.trends_df.ffill()
                # Handle any remaining NaNs at the start with backward fill
                self.price_df = self.price_df.bfill()
                self.trends_df = self.trends_df.bfill()
            elif strategy == 'backward_fill':
                self.price_df = self.price_df.bfill()
                self.trends_df = self.trends_df.bfill()
            elif strategy == 'drop':
                self.price_df = self.price_df.dropna()
                self.trends_df = self.trends_df.dropna()
            else:
                raise ValueError(f"Unknown strategy: {strategy}")
            
            self.logger.info(f"Missing values handled using strategy: {strategy}")
            return missing_counts
            
        except Exception as e:
            self.logger.error(f"Error handling missing values: {str(e)}")
            raise
    
    def handle_extreme_values(self, df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        """
        Handle infinite and extreme values in the specified columns.
        
        Args:
            df: DataFrame to process
            cols: List of column names to check
            
        Returns:
            DataFrame with extreme values handled
        """
        df = df.copy()
        
        for col in cols:
            # Replace infinite values with NaN
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            
            # Calculate robust statistics for outlier handling
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 3 * iqr
            upper_bound = q3 + 3 * iqr
            
            # Cap extreme values
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
            
        return df

    def handle_extreme_values(self, df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        """
        Handle infinite and extreme values in the specified columns.
        
        Args:
            df: DataFrame to process
            cols: List of column names to check
            
        Returns:
            DataFrame with extreme values handled
        """
        df = df.copy()
        
        for col in cols:
            if col in df.columns:
                # Replace infinite values with NaN
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)
                
                # Get non-NaN values for calculating quantiles
                valid_values = df[col].dropna()
                if not valid_values.empty:
                    # Calculate robust statistics for outlier handling
                    q1 = valid_values.quantile(0.25)
                    q3 = valid_values.quantile(0.75)
                    iqr = q3 - q1
                    lower_bound = q1 - 3 * iqr
                    upper_bound = q3 + 3 * iqr
                    
                    # Cap extreme values
                    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
                
                # Fill any remaining NaNs with median
                df[col] = df[col].fillna(valid_values.median())
        
        return df

    def scale_features(self, exclude_cols: List[str] = None) -> None:
        """
        Handle extreme values and scale numerical features.
        
        Args:
            exclude_cols: List of columns to exclude from scaling
        """
        try:
            if exclude_cols is None:
                exclude_cols = ['Date', 'date', 'Price_up_tomorrow', 'is_weekend']
            
            # Get columns to scale for each dataset
            price_num_cols = self.price_df.select_dtypes(include=[np.number]).columns
            price_cols_to_scale = [col for col in price_num_cols if col not in exclude_cols]
            
            trends_num_cols = self.trends_df.select_dtypes(include=[np.number]).columns
            trends_cols_to_scale = [col for col in trends_num_cols if col not in exclude_cols]
            
            # Handle extreme values first
            self.price_df = self.handle_extreme_values(self.price_df, price_cols_to_scale)
            self.trends_df = self.handle_extreme_values(self.trends_df, trends_cols_to_scale)
            
            # Now scale the features
            for cols_to_scale, df in [(price_cols_to_scale, self.price_df), 
                                    (trends_cols_to_scale, self.trends_df)]:
                if cols_to_scale:
                    scaler = StandardScaler()
                    df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
            
            self.logger.info("Features scaled successfully")
            
        except Exception as e:
            self.logger.error(f"Error scaling features: {str(e)}")
            raise
    
    def prepare_final_dataset(self) -> pd.DataFrame:
        """
        Prepare the final dataset by running the complete preprocessing pipeline.
        
        Returns:
            Preprocessed DataFrame ready for modeling
        """
        try:
            # Clean trends data first
            self.clean_trends_data()
            self.logger.info("Trends data cleaned")
            
            # Validate and align timestamps
            valid, message = self.validate_timestamps()
            if not valid:
                raise ValueError(message)
            self.logger.info(message)
            
            # Handle missing values before feature engineering
            initial_missing = self.handle_missing_values(strategy='forward_fill')
            self.logger.info(f"Initial missing values handled: {initial_missing}")
            
            # Add all features
            self.add_technical_indicators()
            self.add_price_patterns()
            self.add_trends_features()
            
            # Handle any new missing values from feature engineering
            final_missing = self.handle_missing_values(strategy='forward_fill')
            self.logger.info(f"Feature engineering missing values handled: {final_missing}")
            
            # Scale features
            self.scale_features()
            
            # Merge datasets
            final_df = pd.merge(
                self.price_df,
                self.trends_df,
                left_on='Date',
                right_on='date',
                how='inner'
            )
            
            # Clean up duplicate date column
            if 'date' in final_df.columns:
                final_df.drop('date', axis=1, inplace=True)
            
            self.logger.info(f"Final dataset prepared with shape: {final_df.shape}")
            return final_df
            
        except Exception as e:
            self.logger.error(f"Error preparing final dataset: {str(e)}")
            raise

The next block initializes the pipeline and uses it to preprocess and merge my data:

In [5]:
# Initialize the pipeline
pipeline = PreprocessingPipeline(price_df=btc_data_cleaned_loaded, trends_df=trends_data_loaded)

# Prepare the final dataset
preprocessed_df = pipeline.prepare_final_dataset()

# Check the results
print("\nFinal dataset shape:", preprocessed_df.shape)
print("\nFinal dataset date range:", preprocessed_df['Date'].min(), "to", preprocessed_df['Date'].max())
print("\nFeatures with missing values:")
print(preprocessed_df.isna().sum()[preprocessed_df.isna().sum() > 0])

INFO:__main__:Trends data cleaned successfully
INFO:__main__:Trends data cleaned
INFO:__main__:Timestamps aligned from 2015-01-23 00:00:00 to 2024-10-26 00:00:00
INFO:__main__:Missing values handled using strategy: forward_fill
INFO:__main__:Initial missing values handled: {'Date': 0, 'Close': 0, 'pct_change_1d': 1, 'sma_3d': 2, 'volatility_3d': 3, 'Price_up_tomorrow': 0, 'date': 0, 'bitcoin wallet': 0, 'crypto wallet': 0, 'buy crypto': 0, 'bitget': 0, 'buy bitcoin': 0}
INFO:__main__:Technical indicators added successfully
INFO:__main__:Price patterns added successfully
INFO:__main__:Trends features added successfully
INFO:__main__:Missing values handled using strategy: forward_fill
INFO:__main__:Feature engineering missing values handled: {'Date': 0, 'Close': 0, 'pct_change_1d': 0, 'sma_3d': 0, 'volatility_3d': 0, 'Price_up_tomorrow': 0, 'rsi_14': 13, 'macd': 0, 'macd_signal': 0, 'macd_hist': 0, 'bb_middle': 19, 'bb_upper': 19, 'bb_lower': 19, 'bb_distance': 19, 'momentum_3d': 3, 'mom


Final dataset shape: (3565, 40)

Final dataset date range: 2015-01-23 00:00:00 to 2024-10-26 00:00:00

Features with missing values:
Series([], dtype: int64)


The pipeline has successfully processed the data. This is a summary of what I now have (`final_df`):

1. Date Coverage:
- Start: 2015-01-23
- End: 2024-10-26
- Total rows: 3,565 days of data

2. Feature Count:
- 40 features total including:
  - Original price features
  - Technical indicators
  - Google Trends features
  - Engineered features

3. Data Quality:
- No missing values in final dataset
- Extreme values have been handled
- Features are properly scaled

### Feature reduction to avoid multicollinearity

Since I have an abundance of features (40 total), I want to do some correlation analysis to check if any of the features are redundant (and removable). Highly correlated features can lead to multicollinearity issues and create unneeded complexity.

Therefore, I will proceed on a series of feature reduction rounds until I end up with a dataset which is reasonablly non-redundant:

In [6]:
# Initial correlation analysis
def analyze_correlations(df, threshold=0.8):
    """Analyze initial feature correlations."""
    feature_cols = [col for col in df.columns if col not in ['Date', 'Price_up_tomorrow']]
    correlations = df[feature_cols].corr()
    
    high_corr_pairs = []
    for i in range(len(correlations.columns)):
        for j in range(i):
            if abs(correlations.iloc[i, j]) > threshold:
                high_corr_pairs.append((
                    correlations.columns[i],
                    correlations.columns[j],
                    correlations.iloc[i, j]
                ))
    
    print(f"\nDataset shape: {df.shape}")
    print(f"\nHighly correlated feature pairs (|correlation| > {threshold}):")
    for pair in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:10]:
        print(f"{pair[0]} vs {pair[1]}: {pair[2]:.3f}")
    
    return high_corr_pairs

# Run initial analysis
initial_correlations = analyze_correlations(preprocessed_df)


Dataset shape: (3565, 40)

Highly correlated feature pairs (|correlation| > 0.8):
sma_3d vs Close: 0.999
bb_upper vs bb_middle: 0.997
bitget_sma_7d vs bitget_sma_3d: 0.996
bb_lower vs bb_middle: 0.996
bitget_sma_3d vs bitget: 0.996
bb_middle vs sma_3d: 0.995
bb_upper vs sma_3d: 0.994
bb_middle vs Close: 0.994
crypto wallet_sma_7d vs crypto wallet_sma_3d: 0.994
bb_upper vs Close: 0.993


The first round of feature reduction focuses on removing technically redundant features. Here, I remove features that are direct mathematical derivatives of other features. This step eliminates pure mathematical redundancies while preserving the underlying market signals:

In [7]:
# First round of feature reduction based on technical indicators
features_to_remove = [
    'sma_3d',           # Highly correlated with Close (0.999)
    'bb_middle',        # Highly correlated with Close (0.994) and other BB bands
    'bb_upper',         # Highly correlated with bb_middle (0.997)
    'bb_lower',         # Highly correlated with bb_middle (0.996)
]

# Create reduced feature set
reduced_df = preprocessed_df.drop(columns=features_to_remove)

# Check remaining correlations
analyze_correlations(reduced_df)


Dataset shape: (3565, 36)

Highly correlated feature pairs (|correlation| > 0.8):
bitget_sma_7d vs bitget_sma_3d: 0.996
bitget_sma_3d vs bitget: 0.996
crypto wallet_sma_7d vs crypto wallet_sma_3d: 0.994
crypto wallet_sma_3d vs crypto wallet: 0.992
bitget_sma_7d vs bitget: 0.991
buy crypto_sma_7d vs buy crypto_sma_3d: 0.990
buy crypto_sma_3d vs buy crypto: 0.989
bitcoin wallet_sma_7d vs bitcoin wallet_sma_3d: 0.986
crypto wallet_sma_7d vs crypto wallet: 0.983
bitcoin wallet_sma_3d vs bitcoin wallet: 0.978


[('macd_signal', 'macd', 0.9536330256635405),
 ('bb_distance', 'rsi_14', 0.8661226168481504),
 ('momentum_14d', 'rsi_14', 0.8932315449736723),
 ('buy crypto', 'crypto wallet', 0.937377690687575),
 ('bitget', 'crypto wallet', 0.8373684502224459),
 ('buy bitcoin', 'bitcoin wallet', 0.8695556526665378),
 ('bitcoin wallet_sma_3d', 'bitcoin wallet', 0.97825307352026),
 ('bitcoin wallet_sma_3d', 'buy bitcoin', 0.8653465228909673),
 ('bitcoin wallet_sma_7d', 'bitcoin wallet', 0.9596358717356744),
 ('bitcoin wallet_sma_7d', 'buy bitcoin', 0.8472777069441694),
 ('bitcoin wallet_sma_7d', 'bitcoin wallet_sma_3d', 0.9858660685737025),
 ('crypto wallet_sma_3d', 'crypto wallet', 0.9919966934859538),
 ('crypto wallet_sma_3d', 'buy crypto', 0.9326463194971262),
 ('crypto wallet_sma_3d', 'bitget', 0.8423642376225172),
 ('crypto wallet_sma_7d', 'crypto wallet', 0.9829462477120543),
 ('crypto wallet_sma_7d', 'buy crypto', 0.9229741372137605),
 ('crypto wallet_sma_7d', 'bitget', 0.8459334329616792),
 ('cr

Upon analyzing the remaining correlations, significant redundancy in the search term features is evident. This is particularly so between raw values and their moving averages.

Since the moving averages are derived from the raw values and don't add significant new information, we can remove them while keeping the more immediate signals from the raw search term data:

In [8]:
# Remove redundant search term features
features_to_remove_2 = [
    'bitget_sma_3d',           # Correlated with bitget (0.996)
    'crypto wallet_sma_3d',     # Correlated with crypto wallet (0.992)
    'buy crypto_sma_3d',        # Correlated with buy crypto (0.989)
    'buy crypto_sma_7d',        # Correlated with buy crypto_sma_3d (0.990)
    'bitcoin wallet_sma_3d',    # Correlated with bitcoin wallet (0.978)
    'bitcoin wallet_sma_7d',    # Correlated with bitcoin wallet_sma_3d (0.986)
    'buy bitcoin_sma_3d',       # Correlated with buy bitcoin (0.963)
    'buy bitcoin_sma_7d'        # Correlated with buy bitcoin_sma_3d (0.972)
]

# Create final feature set
final_reduced_df = reduced_df.drop(columns=features_to_remove_2)

# Check correlations again
analyze_correlations(final_reduced_df)


Dataset shape: (3565, 28)

Highly correlated feature pairs (|correlation| > 0.8):
bitget_sma_7d vs bitget: 0.991
crypto wallet_sma_7d vs crypto wallet: 0.983
macd_signal vs macd: 0.954
buy crypto vs crypto wallet: 0.937
crypto wallet_sma_7d vs buy crypto: 0.923
momentum_14d vs rsi_14: 0.893
buy bitcoin vs bitcoin wallet: 0.870
bb_distance vs rsi_14: 0.866
crypto wallet_sma_7d vs bitget: 0.846
bitget_sma_7d vs crypto wallet_sma_7d: 0.842


[('macd_signal', 'macd', 0.9536330256635405),
 ('bb_distance', 'rsi_14', 0.8661226168481504),
 ('momentum_14d', 'rsi_14', 0.8932315449736723),
 ('buy crypto', 'crypto wallet', 0.937377690687575),
 ('bitget', 'crypto wallet', 0.8373684502224459),
 ('buy bitcoin', 'bitcoin wallet', 0.8695556526665378),
 ('crypto wallet_sma_7d', 'crypto wallet', 0.9829462477120543),
 ('crypto wallet_sma_7d', 'buy crypto', 0.9229741372137605),
 ('crypto wallet_sma_7d', 'bitget', 0.8459334329616792),
 ('bitget_sma_7d', 'crypto wallet', 0.828158611105943),
 ('bitget_sma_7d', 'bitget', 0.9909152666597278),
 ('bitget_sma_7d', 'crypto wallet_sma_7d', 0.8418498925452694),
 ('bitget_momentum', 'bitget', 0.8150173070584644),
 ('bitget_momentum', 'bitget_sma_7d', 0.8120705824067225),
 ('aggregate_interest', 'crypto wallet', 0.840364095489413),
 ('aggregate_interest', 'crypto wallet_sma_7d', 0.8210587187004038)]

The final reduction focuses on the quality and reliability of the remaining features:

In [9]:
# Remove unreliable search terms and redundant features
final_features_to_remove = [
    'aggregate_interest',  # Redundant with individual terms
    'bitget',             # High zero count
    'bitget_momentum',    # Dependent on unreliable base term
    'buy crypto',         # High correlation with crypto_wallet
    'buy crypto_momentum',
    'crypto wallet',      # High zero count
    'crypto wallet_momentum'
]

final_feature_df = final_reduced_df.drop(columns=final_features_to_remove)

# Final correlation check
analyze_correlations(final_feature_df)


Dataset shape: (3565, 21)

Highly correlated feature pairs (|correlation| > 0.8):
macd_signal vs macd: 0.954
momentum_14d vs rsi_14: 0.893
buy bitcoin vs bitcoin wallet: 0.870
bb_distance vs rsi_14: 0.866
bitget_sma_7d vs crypto wallet_sma_7d: 0.842


[('macd_signal', 'macd', 0.9536330256635405),
 ('bb_distance', 'rsi_14', 0.8661226168481504),
 ('momentum_14d', 'rsi_14', 0.8932315449736723),
 ('buy bitcoin', 'bitcoin wallet', 0.8695556526665378),
 ('bitget_sma_7d', 'crypto wallet_sma_7d', 0.8418498925452694)]

#### Concluding feature reduction

After the above feature reduction process, I've arrived at a dataset with 21 features across 3,565 daily observations.

The remaining high correlations (>0.8) in the dataset are justified:
- The strongest correlation exists between MACD and its signal line (0.954), which is expected and necessary as these indicators are designed to work together to generate trading signals.
- The correlation between 14-day momentum and RSI (0.893) is acceptable as they measure momentum in complementary ways, with RSI normalizing the momentum to a 0-100 scale.
- The relationship between "Buy Bitcoin" and "Bitcoin Wallet" search terms (0.870) represents related but distinct user behaviors, capturing different stages of cryptocurrency interest.
- The correlation between Bollinger Band Distance and RSI (0.866) combines volatility and momentum signals, providing complementary market state information.
- The correlation between Bitget SMA and Crypto Wallet SMA (0.842) represents the lowest remaining high correlation, indicating broader market interest patterns.

Since all these correlations are either technically necessary or theoretically justified (and none reach concerning levels for multicollinearity), I can reasonably conclude the feature reduction process here without risking the loss of valuable market signals.

## Splitting the data

Now, it's time to split the dataset into three periods: training, validation, and test.

The validation set (2022-2023) will be used for hyperparameter tuning. The test set (2023-2024) is kept completely separate and used only once at the very end to evaluate model performance.

In [10]:
# Get split points
total_rows = len(final_feature_df)
train_end = '2022-01-01'
val_end = '2023-01-01'

# Create the splits
train_df = final_feature_df[final_feature_df['Date'] < train_end]
val_df = final_feature_df[(final_feature_df['Date'] >= train_end) & 
                         (final_feature_df['Date'] < val_end)]
test_df = final_feature_df[final_feature_df['Date'] >= val_end]

# Print info about splits
print("Data splits:")
print(f"Train: {train_df['Date'].min()} to {train_df['Date'].max()} ({len(train_df)} days)")
print(f"Validation: {val_df['Date'].min()} to {val_df['Date'].max()} ({len(val_df)} days)")
print(f"Test: {test_df['Date'].min()} to {test_df['Date'].max()} ({len(test_df)} days)")

# Create X and y for each split
feature_cols = [col for col in final_feature_df.columns 
                if col not in ['Date', 'Price_up_tomorrow']]

X_train = train_df[feature_cols]
y_train = train_df['Price_up_tomorrow']

X_val = val_df[feature_cols]
y_val = val_df['Price_up_tomorrow']

X_test = test_df[feature_cols]
y_test = test_df['Price_up_tomorrow']

# Print shapes of feature matrices
print("\nFeature matrices shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

# Print class distribution in each split
print("\nClass distribution (percentage of up days):")
print(f"Train: {(y_train == 1).mean():.3f}")
print(f"Validation: {(y_val == 1).mean():.3f}")
print(f"Test: {(y_test == 1).mean():.3f}")

Data splits:
Train: 2015-01-23 00:00:00 to 2021-12-31 00:00:00 (2535 days)
Validation: 2022-01-01 00:00:00 to 2022-12-31 00:00:00 (365 days)
Test: 2023-01-01 00:00:00 to 2024-10-26 00:00:00 (665 days)

Feature matrices shapes:
X_train: (2535, 19)
X_val: (365, 19)
X_test: (665, 19)

Class distribution (percentage of up days):
Train: 0.546
Validation: 0.466
Test: 0.507


## Naive model

In this step, I build a naive model, which simply guesses the next day's bitcoin price movement based on the observed change from the previous to current days. This will be used as a baseline standard for evaluating models built:

In [11]:
# Create naive predictions for each dataset.
# For each day, predict that tomorrow will move in the same direction as today.

from sklearn.metrics import classification_report, confusion_matrix

# Training set naive predictions
naive_train_pred = (train_df['pct_change_1d'] > 0).astype(int)
# Validation set naive predictions
naive_val_pred = (val_df['pct_change_1d'] > 0).astype(int)
# Test set naive predictions
naive_test_pred = (test_df['pct_change_1d'] > 0).astype(int)

print("Naive Model Performance:")
print("\nTraining Set:")
print(classification_report(y_train, naive_train_pred))
print("\nValidation Set:")
print(classification_report(y_val, naive_val_pred))

# Compare confusion matrices
print("\nNaive Model Confusion Matrix:")
print(confusion_matrix(y_val, naive_val_pred))

Naive Model Performance:

Training Set:
              precision    recall  f1-score   support

           0       0.42      0.47      0.45      1152
           1       0.51      0.47      0.49      1383

    accuracy                           0.47      2535
   macro avg       0.47      0.47      0.47      2535
weighted avg       0.47      0.47      0.47      2535


Validation Set:
              precision    recall  f1-score   support

           0       0.54      0.57      0.55       195
           1       0.47      0.43      0.45       170

    accuracy                           0.51       365
   macro avg       0.50      0.50      0.50       365
weighted avg       0.50      0.51      0.50       365


Naive Model Confusion Matrix:
[[112  83]
 [ 97  73]]


As expected, the naive model performs poorly. On the training set, it achieves only 47% accuracy, with similar precision and recall scores around 0.47. Performance on the validation set is marginally better at 51% accuracy, but still essentially random. But, now I can use this to compare against as a baseline.

## Building an XGBoost model

I will train and evaluate an XGBoost model (gradient boosting algorithm that uses decision trees as base learners) since it is known to:

- Handle complex non-linear relationships very well
- Perform effectively on tabular data with mixed feature types

Essentially, my goal is to build a binary classifier which predicts if Bitcoin's price is moving up or down tomorrow. This prediction will be made based on historic Bitcoin and Google Trends data. My goal is to predict the _direction_ of Bitcoin's price, rather than what it will actually be, since this is potentially exploitable for real-life automated trading. I researched SARIMA and SARIMAX time series models, which excell at forecasting based on continous time series data, but they are better suited for predicting actual price values rather than binary directional movement. Additionally, they do not handle multiple input features as well as XGBoost (and I have engineered quite a few features here). Therefore, I have chosen to build an XGBoost model.

First I will create the base model, and then tune its hyperparameters:

### Creating the base model

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Initialize model with balanced classes
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,  # Will adjust based on class distribution
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Set scale_pos_weight based on class distribution
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb.set_params(scale_pos_weight=scale_pos_weight)

# Train
xgb.fit(X_train, y_train)

# Predictions
val_pred = xgb.predict(X_val)
val_pred_proba = xgb.predict_proba(X_val)[:, 1]

# Performance metrics
print("XGBoost Validation Set Performance:")
print("\nClassification Report:")
print(classification_report(y_val, val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nROC AUC Score:", roc_auc_score(y_val, val_pred_proba))

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
})
importance_df = importance_df.sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(importance_df.head(10))

# Compare with previous benchmarks
print("\nModel Comparison on Validation Set:")
print(f"Naive Model Accuracy: 50.68%")
print(f"Random Forest Accuracy: 53.70%")
print(f"XGBoost Accuracy: {(val_pred == y_val).mean():.4%}")

# Check probability distribution and performance at different thresholds
print("\nPerformance at different confidence thresholds:")
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7]

for threshold in thresholds:
    high_conf_preds = (val_pred_proba >= threshold) | (val_pred_proba <= (1-threshold))
    if high_conf_preds.sum() > 0:
        high_conf_accuracy = (y_val[high_conf_preds] == (val_pred_proba[high_conf_preds] >= 0.5)).mean()
        print(f"\nThreshold: {threshold}")
        print(f"Number of predictions: {high_conf_preds.sum()} ({high_conf_preds.sum()/len(y_val)*100:.1f}% of total)")
        print(f"Accuracy: {high_conf_accuracy:.3f}")

c:\Users\David\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:30:22] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Validation Set Performance:

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.68      0.61       195
           1       0.51      0.38      0.43       170

    accuracy                           0.54       365
   macro avg       0.53      0.53      0.52       365
weighted avg       0.53      0.54      0.53       365


Confusion Matrix:
[[133  62]
 [106  64]]

ROC AUC Score: 0.5236500754147814

Top 10 Most Important Features:
                    feature  importance
7               bb_distance    0.064864
3                    rsi_14    0.064322
9               momentum_7d    0.057667
0                     Close    0.057312
15  bitcoin wallet_momentum    0.057205
10             momentum_14d    0.057015
1             pct_change_1d    0.056567
8               momentum_3d    0.056263
5               macd_signal    0.055600
18     buy bitcoin_momentum    0.055562

Model Comparison on Validation Set:
Naive Model Accuracy: 50.6

### Hyperparameter tuning using grid search

I will utilize my computer's GPU to optimize the hyperparameter tuning and training in two steps: first, by conducting a more coarse grid search, and finally, a finer grid search to try to find the optimal hyperparameters for this model:

In [13]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np
from tqdm import tqdm
import itertools

def evaluate_params(params, X_train, y_train, tscv):
    scores = []
    for train_idx, val_idx in tscv.split(X_train):
        X_train_split, X_val_split = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_split, y_val_split = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = XGBClassifier(
            **params,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            tree_method='hist',
            device='cuda',
            eval_metric='logloss'
        )
        
        model.fit(X_train_split, y_train_split)
        pred_proba = model.predict_proba(X_val_split)[:, 1]
        score = roc_auc_score(y_val_split, pred_proba)
        scores.append(score)
    
    return np.mean(scores)

def grid_search(param_grid, X_train, y_train, tscv):
    param_combinations = [dict(zip(param_grid.keys(), v)) 
                         for v in itertools.product(*param_grid.values())]
    
    best_score = 0
    best_params = None
    
    pbar = tqdm(total=len(param_combinations), desc="Testing parameters")
    tested_count = 0
    
    try:
        for params in param_combinations:
            score = evaluate_params(params, X_train, y_train, tscv)
            tested_count += 1
            
            if score > best_score:
                best_score = score
                best_params = params
                print(f"\nNew best score: {best_score:.4f}")
                print(f"Parameters: {best_params}")
                print(f"Progress: {tested_count}/{len(param_combinations)}")
            
            pbar.update(1)
            
    except KeyboardInterrupt:
        print("\nGrid search interrupted. Using best parameters found so far.")
    finally:
        pbar.close()
        
    return best_score, best_params

# Stage 1: Coarse Search
print("Stage 1: Coarse Grid Search")
coarse_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 6],
    'learning_rate': [0.01, 0.1],
    'min_child_weight': [1, 5],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'gamma': [0, 0.2]
}

tscv = TimeSeriesSplit(n_splits=5)
best_coarse_score, best_coarse_params = grid_search(coarse_param_grid, X_train, y_train, tscv)

# Stage 2: Optimized Fine Search
print("\nStage 2: Optimized Fine Grid Search")
fine_param_grid = {
    'n_estimators': [80, 100, 120],  # Centered around best value 100
    'max_depth': [2, 3, 4],          # Centered around best value 3
    'learning_rate': [0.05, 0.1, 0.15],  # Centered around best value 0.1
    'min_child_weight': [1, 2],      # Best was 1, try small increase
    'subsample': [0.7, 0.8],         # Include 0.7 which showed promise
    'colsample_bytree': [0.7, 0.8],  # Include 0.7 which showed promise
    'gamma': [0, 0.1]                # Focus on lower values that performed better
}

# Calculate total combinations
total_combinations = np.prod([len(v) for v in fine_param_grid.values()])
print(f"Total fine-tuning combinations to test: {total_combinations}")

# Run fine grid search
best_fine_score, best_fine_params = grid_search(fine_param_grid, X_train, y_train, tscv)

# Train final model with best parameters
print("\nTraining final model with best parameters...")
best_xgb = XGBClassifier(
    **best_fine_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss'
)

best_xgb.fit(X_train, y_train)

Stage 1: Coarse Grid Search


Testing parameters:   0%|          | 0/32 [00:00<?, ?it/s]c:\Users\David\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:30:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
Testing parameters:   3%|▎         | 1/32 [00:04<02:30,  4.86s/it]


New best score: 0.5452
Parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'gamma': 0}
Progress: 1/32


Testing parameters:   9%|▉         | 3/32 [00:15<02:24,  4.98s/it]


New best score: 0.5457
Parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'gamma': 0}
Progress: 3/32


...


Training final model with best parameters...


c:\Users\David\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The XGBClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=0, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=2, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

### Save the model

In [17]:
# Save the model
best_xgb.save_model('best_xgb.json')

### Evaluation on the test set

Now that the XGBoost model has finished its hyperparameter tuning, it is now time to evaluate the model using evaluation metrics and the test set.

To recap, the training data covered an almost 7-year range, and the validation set which was used during tuning covered data from January 1 to December 31 of 2022.

The test set is an almost 2-year range of data from January 1, 2023 to October 27, 2024.

In [19]:
# Evaluate final model on test set
test_pred = best_xgb.predict(X_test)
test_proba = best_xgb.predict_proba(X_test)

# Calculate naive predictions for test set
naive_test_pred = (test_df['pct_change_1d'] > 0).astype(int)

# Print performance metrics
print("Test Set Performance (2023-2024):")
print("\nNaive Model Confusion Matrix:")
print(confusion_matrix(y_test, naive_test_pred))
print("\nNaive Model Classification Report:")
print(classification_report(y_test, naive_test_pred))

print("\nXGBoost Model Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print("\nXGBoost Model Classification Report:")
print(classification_report(y_test, test_pred))

# Calculate ROC AUC score
roc_auc = roc_auc_score(y_test, test_proba[:, 1])
print(f"\nROC AUC Score: {roc_auc:.3f}")

# Look at high confidence predictions
print("\nPerformance at different confidence thresholds:")
for threshold in [0.5, 0.6, 0.7]:
    high_conf_mask = (test_proba[:, 1] > threshold) | (test_proba[:, 1] < (1-threshold))
    if high_conf_mask.sum() > 0:
        high_conf_accuracy = (y_test[high_conf_mask] == test_pred[high_conf_mask]).mean()
        print(f"\nThreshold: {threshold}")
        print(f"Number of predictions: {high_conf_mask.sum()} ({high_conf_mask.sum()/len(y_test)*100:.1f}% of total)")
        print(f"Accuracy: {high_conf_accuracy:.3f}")

Test Set Performance (2023-2024):

Naive Model Confusion Matrix:
[[170 158]
 [197 140]]

Naive Model Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.52      0.49       328
           1       0.47      0.42      0.44       337

    accuracy                           0.47       665
   macro avg       0.47      0.47      0.47       665
weighted avg       0.47      0.47      0.46       665


XGBoost Model Confusion Matrix:
[[233  95]
 [229 108]]

XGBoost Model Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.71      0.59       328
           1       0.53      0.32      0.40       337

    accuracy                           0.51       665
   macro avg       0.52      0.52      0.49       665
weighted avg       0.52      0.51      0.49       665


ROC AUC Score: 0.542

Performance at different confidence thresholds:

Threshold: 0.5
Number of predictions: 665 (100.0% of tot

#### Analysis

The XGBoost model shows a modest improvement over the naive approach, achieving 51.3% accuracy compared to 47%. Looking at the confusion matrices, there are some interesting patterns.

The naive model is fairly balanced but weak:
- Predicts down days correctly 52% of the time (170/328)
- Predicts up days correctly 42% of the time (140/337)
- Overall similar to (or slightly worse than) random guessing

The XGBoost model shows some improvement but with a notable downward bias:
- Better at predicting down days: 71% accuracy (233/328)
- Worse at predicting up days: 32% accuracy (108/337)
- Makes fewer false up predictions (95 vs 158)
- But misses more actual up days (229 vs 197)

Perhaps the most interesting finding comes from analyzing the model's confidence levels. While the base accuracy across all predictions is 51.3%, the model performs notably better when filtering for high-confidence predictions:
- When looking at predictions with 60% confidence (38.3% of all trades), accuracy improves to 54.1%
- At 70% confidence threshold, accuracy jumps to 60.7%, though this only applies to 8.4% of trading opportunities (about 56 trades over the year)

This suggests a potentially viable trading strategy by only taking trades where the model shows high confidence (>70% threshold). However, the limited number of high-confidence opportunities (only 56 over the entire year) may make this approach impractical for active trading.

The model's ROC AUC score of 0.542 indicates some predictive power above random chance, but the strong bias toward identifying down days (71% recall) versus up days (32% recall) suggests it might be more suitable as a risk management tool than a primary trading signal generator.

### Conclusion

In this milestone, I built and evaluated an XGBoost model to predict Bitcoin's next-day price movements using historical price data and Google Trends features. While the 4.3% accuracy improvement over the naive model is statistically meaningful, its practical significance for trading purposes is limited. The model's strong bias toward predicting down movements (71% accuracy for down days vs 32% for up days) suggests it might be more valuable as a risk management tool than a trading signal generator. The overall accuracy level of 51.3% would likely struggle to overcome trading costs in a real-world scenario.

The most promising finding is the model's 60.7% accuracy on high-confidence predictions, but this only occurs in 8.4% of cases (56 trades/year). Even with sophisticated features and extensive tuning, finding reliably exploitable patterns in Bitcoin price movements remains extremely challenging, with results still very close to random chance.